# Facilitator-Member Difference Identification

In [4]:
import json, pandas as pd
from pathlib import Path
from collections import defaultdict, Counter

# ---- CONFIG ----
DATA_DIR = Path("/Users/maxchalekson/Desktop/gemini_data_analysis/data")   # root with 2021MZT, 2022SLU, ...
OUTPUT_DIR = Path("/Users/maxchalekson/Desktop/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALL_PERSON_SESSION = []   # collect across all conferences
ALL_PERSON_YEAR    = []   # collect across all conferences

# ---- LOADERS ----
def _load_json(fp: Path):
    with open(fp, "r") as f:
        return json.load(f)

def load_conference_data(conf_path: Path):
    """Load session outcomes, person-to-team, and session features_* for a given conference folder."""
    conf_name   = conf_path.name   # e.g., "2021MZT"
    year        = int(conf_name[:4])
    conference  = conf_name[4:]

    outcome          = _load_json(conf_path / f"{conf_name}_outcome.json")
    person_to_team   = _load_json(conf_path / f"{conf_name}_person_to_team.json")
    session_outcomes = _load_json(conf_path / f"{conf_name}_session_outcomes.json")

    # features_*.json (per session)
    features = {}
    for fp in conf_path.glob("features_*.json"):
        sid = fp.stem.replace("features_", "")      # e.g., "2021_09_30_MZT_S5"
        features[sid] = _load_json(fp)
    return year, conference, outcome, person_to_team, session_outcomes, features

# ---- PERSON METRICS FROM TRANSCRIPTS (session_data/*.json) ----
def extract_person_metrics_from_session_data(conf_path: Path, session_id: str):
    """
    Look for /session_data/<session_id>.json (your transcript/annotation file) and
    compute person-level metrics for this session. Return dict: person -> metrics.
    Metrics derived here (prefix p_):
        p_speaking_duration_sec, p_turns, p_interruptions_made, p_overlaps,
        p_screenshare_segments, p_smile_self_total, p_smile_other_total, p_nods_received
    """
    session_file = conf_path / "session_data" / f"{session_id}.json"
    if not session_file.exists():
        return {}

    data = _load_json(session_file)
    # file schema assumed like your example: {"all_speakers": [...], "total_speaking_length": int, "all_data": [ {...}, ... ]}
    rows = data.get("all_data", [])
    by_person = defaultdict(lambda: Counter())

    for r in rows:
        speaker = r.get("speaker")
        if not speaker:
            continue
        dur = r.get("speaking_duration", 0) or 0
        by_person[speaker]["p_speaking_duration_sec"] += float(dur)
        by_person[speaker]["p_turns"] += 1

        # flags (string Yes/No in your example)
        if str(r.get("interuption", "")).strip().lower() == "yes":
            by_person[speaker]["p_interruptions_made"] += 1
        if str(r.get("overlap", "")).strip().lower() == "yes":
            by_person[speaker]["p_overlaps"] += 1
        if str(r.get("screenshare", "")).strip().lower() == "yes":
            by_person[speaker]["p_screenshare_segments"] += 1

        # some numeric affect markers present in your file
        by_person[speaker]["p_smile_self_total"]  += float(r.get("smile_self", 0) or 0)
        by_person[speaker]["p_smile_other_total"] += float(r.get("smile_other", 0) or 0)
        by_person[speaker]["p_nods_received"]     += float(r.get("nods_others", 0) or 0)

    # convert to dict of dict
    out = {}
    for person, cnt in by_person.items():
        out[person] = dict(cnt)
    return out

# ---- BUILDERS ----
def build_person_session(year, conference, conf_path, session_outcomes, features):
    """One row per (person, session), with role_in_session, ctx_* from features, and p_* from transcripts."""
    special = {"missing_names", "people_not_in_any_team"}
    sessions = [sid for sid in session_outcomes.keys() if sid not in special]
    rows = []

    for sid in sessions:
        so = session_outcomes[sid]
        facilitators = set(so.get("facilitators", []) or [])
        speakers     = set(so.get("all_speakers", []) or [])

        # union of all team members listed under 'teams'
        members = set()
        for _, tinfo in (so.get("teams", {}) or {}).items():
            for m in tinfo.get("members", []) or []:
                members.add(m)

        people = facilitators | speakers | members

        # session-level features (context)
        ctx = {f"ctx_{k}": v for k, v in (features.get(sid, {}) or {}).items()}

        # person-level from transcripts (if available)
        person_metrics = extract_person_metrics_from_session_data(conf_path, sid)

        for person in sorted(people):
            if person in facilitators:
                role_in_session = "facilitator"
            elif person in members:
                role_in_session = "member"
            elif person in speakers:
                role_in_session = "participant"
            else:
                role_in_session = "unknown"

            pmet = person_metrics.get(person, {})  # p_* keys
            rows.append({
                "person_name": person,
                "conference": conference,
                "year": year,
                "session_id": sid,
                "role_in_session": role_in_session,
                **ctx,
                **pmet
            })

    return pd.DataFrame(rows)

def build_person_year(person_session_df, person_to_team):
    """Aggregate to one row per (person, conference, year): role tallies, team counts, mean ctx_* and mean/sum p_*."""
    if person_session_df.empty:
        return person_session_df

    # Role tallies per person-year
    pivot = (person_session_df
             .pivot_table(index=["person_name","conference","year"],
                          columns="role_in_session",
                          values="session_id",
                          aggfunc="nunique",
                          fill_value=0)
             .reset_index())
    for col in ["facilitator","member","participant","unknown"]:
        if col not in pivot.columns:
            pivot[col] = 0
    pivot["sessions_total"] = pivot["facilitator"] + pivot["member"] + pivot["participant"] + pivot["unknown"]

    # Primary role: facilitator > member > participant > unknown
    role_priority = {"facilitator": 3, "member": 2, "participant": 1, "unknown": 0}
    def primary_role(row):
        # break ties by priority order implicitly
        best = max(role_priority, key=lambda r: (row.get(r,0)>0, role_priority[r]))
        return best
    pivot["role_primary"] = pivot.apply(primary_role, axis=1)

    # Mean of ctx_* and p_* across sessions the person attended
    metric_cols = [c for c in person_session_df.columns if c.startswith("ctx_") or c.startswith("p_")]
    if metric_cols:
        agg = (person_session_df
               .groupby(["person_name","conference","year"], as_index=False)[metric_cols]
               .mean())
        out = pivot.merge(agg, on=["person_name","conference","year"], how="left")
    else:
        out = pivot

    # Team outcomes from person_to_team
    team_rows = []
    for pname in out["person_name"]:
        lst = person_to_team.get(pname, [])
        funded = sum(1 for t in lst if t.get("funded_status", 0) == 1)
        unfund = sum(1 for t in lst if t.get("funded_status", 0) == 0)
        team_rows.append((pname, funded, unfund, ", ".join([t.get("team_id","") for t in lst])))
    team_df = pd.DataFrame(team_rows, columns=["person_name","teams_funded","teams_unfunded","team_ids"])
    team_df["teams_total"] = team_df["teams_funded"] + team_df["teams_unfunded"]

    out = out.merge(team_df, on="person_name", how="left")
    return out

# ---- DRIVER: build & COMBINE EVERYTHING INTO ONE FILE ----
for conf_path in sorted(DATA_DIR.iterdir()):
    if not conf_path.is_dir():
        continue
    conf_name = conf_path.name
    core = conf_path / f"{conf_name}_session_outcomes.json"
    if not core.exists():
        continue

    year, conference, outcome, person_to_team, session_outcomes, features = load_conference_data(conf_path)
    ps = build_person_session(year, conference, conf_path, session_outcomes, features)
    py = build_person_year(ps, person_to_team)

    # stash into combined collectors
    if not ps.empty:
        ALL_PERSON_SESSION.append(ps)
    if not py.empty:
        ALL_PERSON_YEAR.append(py)

# Combine & write ONE spreadsheet for person-year
if ALL_PERSON_YEAR:
    all_py = pd.concat(ALL_PERSON_YEAR, ignore_index=True).sort_values(
        ["person_name","year","conference"]
    )
    out_path = OUTPUT_DIR / "ALL_person_year.csv"
    all_py.to_csv(out_path, index=False)
    print(f"Wrote ONE combined file: {out_path}  ({len(all_py)} rows)")
else:
    print("No person-year rows produced.")

# Optional: also write ONE combined person-session file (comment out if not needed)
# if ALL_PERSON_SESSION:
#     all_ps = pd.concat(ALL_PERSON_SESSION, ignore_index=True).sort_values(
#         ["person_name","year","conference","session_id"]
#     )
#     out_path_ps = OUTPUT_DIR / "ALL_person_session.csv"
#     all_ps.to_csv(out_path_ps, index=False)
#     print(f"Wrote combined person-session file: {out_path_ps}  ({len(all_ps)} rows)")

Wrote ONE combined file: /Users/maxchalekson/Desktop/outputs/ALL_person_year.csv  (790 rows)
